In [1]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(patchwork)

seurat_merged <- readRDS("../data/processed/seurat_merged_pca.rds")
dir.create("../results/clustering", showWarnings = FALSE)

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
seurat_merged <- FindNeighbors(seurat_merged, dims = 1:15)
seurat_merged <- FindClusters(seurat_merged, resolution = 0.5)

table(Idents(seurat_merged))

Computing nearest neighbor graph

Computing SNN



Modularity Optimizer version 1.3.0 by Ludo Waltman and Nees Jan van Eck

Number of nodes: 19772
Number of edges: 681553

Running Louvain algorithm...
Maximum modularity in 10 random starts: 0.9059
Number of communities: 16
Elapsed time: 3 seconds



   0    1    2    3    4    5    6    7    8    9   10   11   12   13   14   15 
3169 2769 2433 2286 2223 1902 1857  999  732  459  451  171  100   99   69   53 

In [3]:
seurat_merged <- RunUMAP(seurat_merged, dims = 1:15)

pdf("../results/clustering/01_UMAP_overview.pdf", width = 14, height = 6)
p1 <- DimPlot(seurat_merged, reduction = "umap", label = TRUE) + ggtitle("Clusters")
p2 <- DimPlot(seurat_merged, reduction = "umap", group.by = "sample_id") + ggtitle("Sample ID")
p1 + p2
dev.off()

Warning message:
“The default method for RunUMAP has changed from calling Python UMAP via reticulate to the R-native UWOT using the cosine metric
To use Python UMAP via reticulate, set umap.method to 'umap-learn' and metric to 'correlation'
This message will be shown once per session”


18:40:41 UMAP embedding parameters a = 0.9922 b = 1.112

18:40:41 Read 19772 rows and found 15 numeric columns

18:40:41 Using Annoy for neighbor search, n_neighbors = 30

18:40:41 Building Annoy index with metric = cosine, n_trees = 50

0%   10   20   30   40   50   60   70   80   90   100%

[----|----|----|----|----|----|----|----|----|----|

*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
|

18:40:43 Writing NN index file to temp file /tmp/RtmpeupVLp/filef875e2c0fbe

18:40:43 Searching Annoy index using 1 thread, search_k = 3000

18:40:48 Annoy recall = 100%

18:40:48 Commencing smooth kNN distance calibration using 1 thread
 with target n_neighbors = 30

18:40:49 Initializing from normalized Laplacian + noise (using RSpectra)

18:40:50 Commencing optimization for 200 epochs, with 831142 positive edges

18:40:50 Using rng type: pcg

18:40:56 Optimization finished



agg_record_f87709d19ca 
                     2

In [4]:
seurat_merged <- JoinLayers(seurat_merged)

markers <- FindAllMarkers(seurat_merged, only.pos = TRUE, min.pct = 0.25, logfc.threshold = 0.25)
write.csv(markers, "../results/clustering/all_cluster_markers.csv", row.names = FALSE)

top_markers <- markers %>% group_by(cluster) %>% slice_max(n = 5, order_by = avg_log2FC)
write.csv(top_markers, "../results/clustering/top5_markers_per_cluster.csv", row.names = FALSE)
print(top_markers, n = 100)

Calculating cluster 0

For a (much!) faster implementation of the Wilcoxon Rank Sum Test,
(default method for FindMarkers) please install the presto package
--------------------------------------------
install.packages('devtools')
devtools::install_github('immunogenomics/presto')
--------------------------------------------
After installation of presto, Seurat will automatically use the more 
efficient implementation (no further action necessary).
This message will be shown once per session

Calculating cluster 1

Calculating cluster 2

Calculating cluster 3

Calculating cluster 4

Calculating cluster 5

Calculating cluster 6

Calculating cluster 7

Calculating cluster 8

Calculating cluster 9

Calculating cluster 10

Calculating cluster 11

Calculating cluster 12

Calculating cluster 13

Calculating cluster 14

Calculating cluster 15



# A tibble: 80 × 7
# Groups:   cluster [16]
       p_val avg_log2FC pct.1 pct.2 p_val_adj cluster gene     
       <dbl>      <dbl> <dbl> <dbl>     <dbl> <fct>   <chr>    
 1 0               2.36 0.317 0.07  0         0       CHRM3-AS2
 2 0               2.02 0.337 0.087 0         0       NELL2    
 3 0               1.95 0.426 0.114 0         0       TRABD2A  
 4 0               1.81 0.56  0.171 0         0       CCR7     
 5 8.34e-222       1.79 0.291 0.091 1.67e-217 0       TSHZ2    
 6 9.46e-242       2.24 0.252 0.061 1.89e-237 1       LMNA     
 7 1.82e-237       2.10 0.256 0.063 3.64e-233 1       ICOS     
 8 4.79e-319       2.08 0.347 0.089 9.58e-315 1       AQP3     
 9 2.42e-221       2.08 0.271 0.076 4.85e-217 1       TBC1D4   
10 5.36e-250       1.88 0.37  0.124 1.07e-245 1       CORO1B   
11 0               3.97 0.472 0.081 0         2       PTGDS    
12 0               3.79 0.285 0.029 0         2       KLRC2    
13 0               3.01 0.304 0.049 0         2       LINC02

In [7]:
pbmc_markers <- c("CD3D","CD3E","CD4","IL7R","CD8A","CD8B","CCL5","CD79A","GNLY","NKG7","KLRD1","MS4A1",  
              "CD14","LYZ","S100A8","FCGR3A","MS4A7","FCER1A","CST3","PPBP","PF4")

pdf("../results/clustering/02_canonical_markers.pdf", width = 16, height = 12)
FeaturePlot(seurat_merged, features = pbmc_markers, ncol = 4)
DotPlot(seurat_merged, features = pbmc_markers) + RotatedAxis()
dev.off()

agg_record_f877a662de6 
                     2

In [12]:
seurat_merged@meta.data %>%
  group_by(seurat_clusters) %>%
  summarise(median_nFeature = median(nFeature_RNA), median_nCount = median(nCount_RNA), n = n())

seurat_clusters,median_nFeature,median_nCount,n
<fct>,<dbl>,<dbl>,<int>
0,1405.0,5490.0,3169
1,1639.0,5508.0,2769
2,1454.0,3675.0,2433
3,1446.0,4273.0,2286
4,1671.0,4264.0,2223
5,1623.0,5292.5,1902
6,1450.0,3461.0,1857
7,1346.0,3969.0,999
8,2593.0,8584.5,732


High expression of CD3D, CD3E, IL7R in cluster 1 and 5 suggest it's possibility of being CD4 T cells.

CD8A, CD8B along with CD3 markers in cluster 0, 3, 4 can be due to them being CD4 T cells.

In cluster 2 and 6, there was seen high expression of CCL5, GNLy, NKG7 which suggest them being NK cells.
CD79A and MS4A1 in cluster 7 and 10 can indicate them being B cells.

CD14, LYZ expression in clusters 8 and 11 can refer to them being CD14 monocytes.

FCGR3A non classical monocytes, which requires high expression of FCGR3A and MS4A7 was found in cluster 12.

PPBP, PF4 high expression can be implied as cluster 15 being platelets.

cluster 9, 13 and 14 couldn't be distinguished by the canonical markers, so checked the top 5 marker genes of those clusters. 

High expression PCNA, SMC4, STMN1, FABP5 shows it's possiblity of being proliferating T/NK cells

PAX5, FCRL1, FCRL5 overexpression can refer to cluster 14 being memory B cells.

Genes expressed in cluster 13, ASIP, SCT, SHD don't belong in the PBMC bioogy. and 0 p- values in those genes along with highest nCount and nFeature of all clusters can be read as technical arifact, likely doublets or RNA contamination. 

In [15]:
new_ids <- c(
  "0" = "CD8 T cells",
  "1" = "CD4 T cells",
  "2" = "NK cells",
  "3" = "CD8 T cells",
  "4" = "CD8 T cells",
  "5" = "CD4 T cells",
  "6" = "NK cells",
  "7" = "B cells",
  "8" = "CD14 Monocytes",
  "9" = "Proliferating T/NK cells",
  "10" = "B cells",
  "11" = "CD14 Monocytes",
  "12" = "FCGR3A Monocytes",
  "13" = "Unknown",
  "14" = "Memory B cells",
  "15" = "Platelets"
  # ... continue for every cluster number in your table(Idents(seurat_merged)) output
)

seurat_merged <- RenameIdents(seurat_merged, new_ids)
seurat_merged$cell_type <- Idents(seurat_merged)

ERROR: Error in RenameIdents.Seurat(seurat_merged, new_ids): Cannot find any of the provided identities


In [10]:
pdf("../results/clustering/03_UMAP_annotated.pdf", width = 10, height = 8)
DimPlot(seurat_merged, reduction = "umap", label = TRUE, repel = TRUE) + ggtitle("Annotated Cell Types")
dev.off()

cell_counts <- table(seurat_merged$cell_type, seurat_merged$sample_id)
write.csv(as.data.frame.matrix(cell_counts), "../results/clustering/cell_type_counts_by_sample.csv")

agg_record_f872fcc2530 
                     2

In [11]:
saveRDS(seurat_merged, "../data/processed/seurat_annotated.rds")
sessionInfo()

R version 4.6.1 (2026-06-24)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 24.04.5 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/blas/libblas.so.3.12.0 
LAPACK: /usr/lib/x86_64-linux-gnu/lapack/liblapack.so.3.12.0  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Asia/Dhaka
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] future_1.69.0      patchwork_1.3.2    ggplot2_4.0.2      dplyr_1.2.1       
[5] Seurat_5.5.1       SeuratObject_5.4.0 sp_2.2-3          

loaded via a namespace (and not attached):
  [1] deldir_